In [0]:
# Instalar bibliotecas necessárias
# No Databricks use: %pip install azure-storage-blob pandas pyarrow python-dotenv
# No Jupyter local use: !pip install azure-storage-blob pandas pyarrow python-dotenv

%pip install azure-storage-blob pandas pyarrow python-dotenv


In [0]:
import os
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io
import random
import datetime
from dotenv import load_dotenv

# Carregar variáveis do .env (para ambiente local)
load_dotenv()

# Tenta usar dbutils (Databricks). Se não existir, cai para variável de ambiente
try:
    connection_string = dbutils.secrets.get(scope="kvfiaptechprod", key="AZURE-STORAGE-CONNECTION")
except NameError:
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION")

# Verifica se conseguiu recuperar
if not connection_string:
    raise ValueError("❌ Connection string não encontrada. Configure no Key Vault (Databricks) ou no arquivo .env local.")

# Nome do container bronze
container_name = "bronze"

# Criar cliente de serviço
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
print("✅ Conexão com Azure Storage estabelecida")


In [0]:
# Função que gera um registro fictício baseado na tabela INEP Alunos
def gerar_registro():
    return {
        "ano": random.choice([2023, 2024]),
        "id_municipio": str(random.randint(1000000, 9999999)),
        "id_escola": f"E{random.randint(1000,9999)}",
        "id_aluno": f"A{random.randint(100000,999999)}",
        "proficiencia": round(random.uniform(100, 300), 2),
        "data_ingestao": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }


In [0]:
# Gerar um lote de 100 registros simulados
df = pd.DataFrame([gerar_registro() for _ in range(100)])

# Visualizar os primeiros registros
print("👀 Visualização dos dados simulados:")
display(df.head())

# Converter para Parquet em memória
parquet_buffer = io.BytesIO()
df.to_parquet(parquet_buffer, index=False, engine="pyarrow")

# Nome do arquivo com partição por data
data_particao = datetime.datetime.now().strftime("%Y/%m/%d")
blob_name = f"inep_alunos_simulado/{data_particao}/dados.parquet"

# Upload para o container bronze
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)

print(f"✅ Dados simulados gravados no bronze em {blob_name}")


In [0]:
import time

# Simular streaming: gerar e gravar novos dados a cada 5 segundos
for i in range(5):  # número de ciclos
    df = pd.DataFrame([gerar_registro() for _ in range(10)])  # 10 registros por ciclo
    
    # Visualizar os primeiros registros de cada lote
    print(f"👀 Lote {i+1} - preview:")
    display(df.head())
    
    parquet_buffer = io.BytesIO()
    df.to_parquet(parquet_buffer, index=False, engine="pyarrow")
    
    data_particao = datetime.datetime.now().strftime("%Y/%m/%d/%H%M%S")
    blob_name = f"inep_alunos_streaming/{data_particao}/dados.parquet"
    
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
    blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)
    
    print(f"📤 Lote {i+1} enviado para {blob_name}")
    time.sleep(5)  # espera 5 segundos antes do próximo lote
